# Chapter 14: Doing the Right Thing
*From: Designing Data-Intensive Applications*

---

## TL;DR

- **Data systems have moral consequences.** Every architecture we build for a purpose also has unintended downstream effects on the people whose data flows through it; engineers share responsibility for those effects.
- **Predictive analytics encodes and amplifies bias.** Models extrapolate from the past; if the past is discriminatory, the model laundries that discrimination behind a veneer of mathematical rigor.
- **Tracking is surveillance.** Swap the word "data" for "surveillance" in your architecture docs — if the phrases suddenly sound sinister, you have built a surveillance system and should own that framing.
- **Privacy is a decision right, not a secret.** It is the freedom to choose what to reveal to whom; click-wrap consent and buried privacy policies cannot meaningfully transfer that right.
- **Data is a toxic asset.** It leaks, gets subpoenaed, gets sold in bankruptcy, outlives the regimes that collected it. Data you do not have cannot be abused.
- **Data minimization is the engineering response.** Collect less, retain less, purge aggressively, and self-regulate before the regulators do.

---

## Why This Chapter Exists

Every other chapter in this book evaluates architectures on their ability to be reliable, scalable, and maintainable. Those are necessary properties, but they are not sufficient. A system can be perfectly linearizable, infinitely scalable, and exquisitely maintainable while also systematically excluding people from employment, insurance, and housing; normalizing mass surveillance; and concentrating power in the collectors of data. The engineering does not absolve the engineers.

The book closes on ethics precisely because the preceding 13 chapters give us dangerous capabilities — distributed transactions, stream processing, replicated state, vector search, machine-learned decision systems — whose combined reach is historically unprecedented. The comparison the chapter reaches for is the Industrial Revolution: enormous productive power that took generations to tame with environmental protection, labor law, and food safety. Our information age is still in its early, unregulated decades. What follows is not a compliance checklist. It is a participatory, iterative practice of reflection, dialog with the people affected, and accountability for outcomes.

---

## 1. Algorithmic Decision-Making and Bias

*Pages 609-612 — Predictive Analytics, Bias and Discrimination, Responsibility and Accountability.*

Wherever an organization wants to avoid a costly mistake — a bad loan, an ineffective hire, a hijacking, a reoffender — a predictive model can be trained to say "no" more cheaply and at higher scale than a human reviewer. From the firm's point of view, a missed opportunity is a rounding error and a catastrophic yes is expensive, so caution compounds. But the individual on the receiving end may be labeled risky by many such systems at once, systematically excluded from jobs, air travel, insurance, rental housing, and financial services — what Bill Davidow called an "algorithmic prison." Criminal courts presume innocence; automated systems do not.

The hope for data-driven decisions was that they would be more fair than a biased human reviewer. In practice, models do not replace the decision rule — they infer it from historical data. If the training data encodes discrimination, the model learns and amplifies the discrimination, opaquely. Anti-discrimination law forbids treating people differently on the basis of protected traits like race, but a model can reach the same outcome using correlated features: a postal code or even an IP address is a strong predictor of race in a segregated city. Cegłowski's summary is that "machine learning is like money laundering for bias."

A traditional credit score summarizes *how you behaved in the past*. ML-driven scoring summarizes *who is similar to you, and how did people like you behave*. The second question is stereotyping. When the model gets an individual wrong — puts them in the wrong bucket, acts on erroneous data — recourse is almost impossible: the decision is probabilistic, opaque, and accountable to no one. People should not be able to evade responsibility by blaming the algorithm.

```mermaid
graph LR
  subgraph Inputs[Input features]
    Postal[Postal code]
    Browser[Browser / IP]
    Purchases[Purchase history]
    Education[Schools attended]
  end
  subgraph Protected[Protected traits<br/>legally off-limits]
    Race[Race]
    Class[Socioeconomic class]
  end
  Inputs -- correlated with --> Protected
  Inputs --> Model[(ML scoring model)]
  Model --> Score[Risk score]
  Score --> Decision{Loan / hire / bail?}
  Decision -- No --> Exclude[Systemic exclusion]
  Exclude -. appeal? .-> Opaque[Opaque rationale<br/>no recourse]
```

### Human vs. algorithmic decisions

| Dimension | Human reviewer | ML-scored decision |
|---|---|---|
| Bias source | Personal prejudice, fatigue, mood | Training data + feature correlations |
| Scale | One at a time | Millions per day |
| Appeal | Explain case, ask supervisor | Usually none; rationale opaque |
| Explainability | Narrative reasons | Feature weights, often proprietary |
| Uniformity | Varies by reviewer | Identical bias applied uniformly |
| Accountability | The person who decided | Diffused: modeller, ops, vendor, firm |

See: [[01-algorithmic-decision-making-and-bias]].

---

## 2. Feedback Loops and Systems Thinking

*Pages 612-613 — Feedback Loops.*

Predictive systems do not sit outside the world they predict on. They change it. The chapter's canonical example: an employer uses credit scores to screen hires. A good worker hits a patch of financial misfortune outside their control, misses payments, and watches their score fall. The lower score now makes them less likely to be hired. Joblessness pushes them toward poverty, which further worsens the score, which further worsens hiring prospects. A downward spiral, camouflaged as mathematical rigor.

Feedback loops are not confined to individuals. Economists studying algorithmic pricing at German gas stations found that once retailers adopted algorithmic pricing, competition softened and consumer prices rose — the pricing algorithms had, without coordinating, learned to collude. Recommendation systems produce echo chambers in which stereotypes, misinformation, and polarization breed, with measurable impact on elections.

The defense against surprise is *systems thinking*: reasoning about the entire system — the computerized parts *and* the people interacting with them — rather than the model in isolation. The key diagnostic question is whether a deployed system tends to amplify existing differences (the rich get richer, the poor poorer) or to counteract injustice. Even with the best intentions, unintended consequences should be the default assumption.

```mermaid
graph LR
  Score[Low credit score] --> Hiring[Employer rejects applicant]
  Hiring --> Joblessness[Continued joblessness]
  Joblessness --> Missed[Missed payments]
  Missed --> Score
  Score -. is used as input to .-> Hiring
  classDef loop fill:#fee,stroke:#a33,stroke-width:2px;
  class Score,Hiring,Joblessness,Missed loop;
```

### A tiny simulation of the credit-score trap

The loop below is a toy — not a calibrated model — but it makes the dynamic concrete. A person starts employed with a middling credit score. A single income shock causes missed payments; the score drops; the drop reduces hiring probability next period; unemployment causes more missed payments; repeat.

In [ ]:
# Feedback loop: credit score -> hireability -> employment -> credit score
# Stdlib-only toy simulation. Not calibrated; illustrates the dynamic only.
import random

random.seed(14)
score = 680            # starting FICO-ish score
employed = True
income_shock_month = 3  # month of a one-time misfortune

print(f"{'month':>5} {'employed':>9} {'score':>6} {'hire_p':>7}")
for month in range(1, 13):
    # Score degrades when unemployed or after a shock; recovers slowly when employed.
    if not employed:
        score -= 25
    elif month == income_shock_month:
        score -= 40  # one-time shock: medical bill, layoff, etc.
    else:
        score = min(score + 3, 850)

    # Hireability is a rough sigmoid in the score.
    hire_p = max(0.05, min(0.95, (score - 500) / 300))
    employed = random.random() < hire_p
    print(f"{month:>5d} {str(employed):>9} {score:>6d} {hire_p:>7.2f}")

See: [[02-feedback-loops-and-systems-thinking]].

---

## 3. Surveillance as a Lens

*Pages 614-616 — Privacy and Tracking, Surveillance.*

Tracking behavioral data is the engine behind many user-facing improvements: click-through data improves search ranking, co-purchase data powers recommendations, A/B tests improve UX. When the data is used to serve the user, the company is performing a service and the user is the customer. When the business model is advertising, the *advertiser* is the customer and the user's data is the product; tracking then becomes more detailed, retention longer, and the service's interests drift away from the user's.

The chapter's thought experiment: take any buzzword sentence — "In our data-driven organization we collect real-time data streams and store them in our data warehouse. Our data scientists use advanced analytics..." — and replace "data" with "surveillance." If the phrase now sounds sinister, that is diagnostic. We have built, almost accidentally, the greatest mass-surveillance infrastructure ever seen: internet-connected microphones in every home (phones, TVs, assistants, toys, baby monitors), location tracking in every pocket, and continuous health telemetry on every wrist.

Why do people accept this? Partly because the purpose seems benign — better recommendations, not political control. Partly because those with nothing to fear under current power structures do not feel the cost. But sensors improve, inference gets sharper (a smartwatch accelerometer can recover typed passwords), and the uses creep: driver-behavior data without consent affects insurance premiums; fitness-tracker coverage becomes a health-insurance condition.

### User-serving tracking vs. surveillance-for-advertising

| Attribute | User-serving tracking | Surveillance-for-advertising |
|---|---|---|
| Customer | The user | The advertiser |
| Data minimality | Only what improves the feature | Maximize detail and scope |
| Retention | As short as the feature needs | Long-term profile building |
| Purpose transparency | Clear to the user | Obscured behind "personalization" |
| Incentive alignment | Aligned with user's goals | User's interests second to ad targeting |

### Historical vs. corporate surveillance

| Dimension | Mid-20th-century state surveillance | Present-day corporate surveillance |
|---|---|---|
| Reach | Where agents were deployed | Everywhere a phone goes |
| Cost per person watched | High, manual | Near zero, automated |
| Participation | Coerced | "Voluntary" via terms of service |
| Data permanence | Paper files, finite | Indefinite, searchable, joinable |
| Stated purpose | Political control | Serving ads, improving engagement |
| Accountability | Subject to law, sometimes | Private; disclosed selectively |

See: [[03-surveillance-as-a-lens]].

---

## 4. Meaningful Consent and Privacy as a Decision Right

*Pages 616-619 — Consent and Freedom of Choice, Privacy and Use of Data.*

The standard industry defense — "users consented, they clicked Accept" — does not survive scrutiny. Most users have no practical understanding of what is collected, how it is joined with third-party datasets, or how long it is retained; privacy policies more often obscure than illuminate. Data from one user reveals things about non-users who never agreed to anything (social graph, location co-occurrence, genetic relations). The exchange is one-sided: the user cannot negotiate which data to provide in return for which service.

The GDPR sets a high bar: consent must be "freely given, specific, informed, and unambiguous," refusable without detriment, and written in clear, accessible language; silence and pre-ticked boxes do not count. But when a service has strong network effects — when it has become "essential for basic social participation" — refusal carries a social cost that makes consent unfree in practice.

Privacy is not the same as secrecy. Having privacy means having the freedom to choose what to reveal to whom in each situation; it is a *decision right* about one's own information. When data is extracted via surveillance, that decision right is not eliminated but *transferred* to the collector: the company now decides who sees what. Companies exercise that right in their own economic interest — revealing intimate inferences indirectly via ad-targeting buckets, keeping the raw extent secret because revealing it would be perceived as creepy and hurt the business.

```mermaid
graph LR
  subgraph Before[Before surveillance]
    U1[User] -- decides --> V1[What to reveal to whom]
  end
  subgraph After[After surveillance]
    U2[User] -- generates data --> C[Collector]
    C -- decides --> V2[What to reveal to whom<br/>to maximize profit]
    U2 -. no longer holds .-> V2
  end
```

### GDPR consent bar vs. common practice

| GDPR requirement | Real-world common practice |
|---|---|
| Freely given (refusable without detriment) | "Accept all or leave the service" |
| Specific (per purpose) | One blanket toggle |
| Informed (plain language) | 6,000-word legal document |
| Unambiguous (active opt-in) | Pre-ticked boxes, inferred from "continuing to use" |
| Withdrawable at any time | Buried, multi-step, sometimes absent |

See: [[04-meaningful-consent-and-privacy]].

---

## 5. Data as a Toxic Asset

*Pages 619-622 — Data as Assets and Power, Remembering the Industrial Revolution.*

Behavioral data is sometimes framed as "data exhaust" — worthless byproduct that analytics heroically recycles into value. The framing is backwards. If targeted advertising pays for the service, then the user activity generating the data is closer to unpaid labor; the application is the lure, and the surveillance infrastructure is the factory. Personal data is a valuable enough asset that an entire data-broker industry operates in near-secrecy to aggregate and resell it, and startup valuations correlate with "eyeballs" — i.e., surveillance reach.

Because data is valuable, many parties want it. Companies collect it. Governments seek it via secret deals, legal compulsion, or theft. Bankrupt companies sell it. Data is hard to secure and breaches happen routinely. That constellation of risks has led critics — Schneier, Scott, Pesce — to reframe data as a *toxic asset*, *hazardous material*, or *the new uranium*: incredibly powerful, but radioactive and durable.

When you collect data today, you are committing not only to this year's political and corporate environment but to every future one: every future management, every future acquirer, every future government. Schneier: "It is poor civic hygiene to install technologies that could someday facilitate a police state." The closest historical analog is the Industrial Revolution, which delivered enormous gains while creating pollution, unsafe workplaces, and child labor — problems that took generations of regulation to constrain. "Data is the pollution problem of the information age."

### Data-as-oil vs. data-as-uranium

| Framing | Implication | What it encourages |
|---|---|---|
| Data as oil | Extract, refine, sell | Maximize collection; build pipelines |
| Data as exhaust | Free byproduct worth recycling | Casual retention; no duty of care |
| Data as labor | Users produce it, should be compensated | Reciprocity; bargaining power |
| Data as toxic asset | Valuable but hazardous; contain and dispose | Minimize, retain briefly, purge |
| Data as uranium | Powerful and dangerous; strict custody | Treat access as a regulated privilege |

### Industrial Revolution parallel

| Industrial age | Information age |
|---|---|
| Factory smoke and water pollution | Data exhaust and surveillance infrastructure |
| Child labor, unsafe workplaces | Dark patterns, non-consensual tracking |
| Adulterated food | Manipulated feeds, misinformation |
| Environmental regulation (decades late) | Data-protection law (GDPR, still catching up) |
| Health inspections | Privacy audits, breach notification |
| Public health crises forced action | Breach scandals and surveillance exposés force action |

See: [[05-data-as-toxic-asset]].

---

## 6. Data Minimization and Engineering Self-Regulation

*Pages 622-624 — Legislation and Self-Regulation.*

GDPR encodes the engineering prescription directly: personal data must be "collected for specified, explicit and legitimate purposes and not further processed in a manner that is incompatible with those purposes" and be "adequate, relevant and limited to what is necessary." This is *data minimization*, and it runs head-on into the default big-data philosophy of collect-everything-now-figure-out-uses-later. The regulation has been weakly enforced and has not yet shifted industry culture at scale.

The chapter's prescription is that culture must shift from within. Stop treating users as metrics to be optimized; they are humans with dignity, agency, and a claim to respect. Self-regulate collection and retention before regulators force the issue. Educate users about what is being collected rather than keeping them in the dark. Treat the individual right to control one's data the way a national park treats the land: if we do not actively protect it, it will be destroyed — the tragedy of the commons applied to privacy.

The concrete operational move: do not retain data forever. Purge as soon as it is no longer needed. Minimize what is collected in the first place. Data you do not have is data that cannot be leaked, stolen, or subpoenaed. If you work in technology and do not consider the societal impact of your work, you are not doing your job.

```mermaid
graph TD
  Need{Do we actually need<br/>this field?}
  Need -- No --> Drop[Don't collect it]
  Need -- Yes --> Purpose{Is the purpose specified,<br/>explicit, legitimate?}
  Purpose -- No --> Drop
  Purpose -- Yes --> Collect[Collect the minimum necessary]
  Collect --> Retain{Still needed for<br/>that purpose?}
  Retain -- No --> Purge[Purge]
  Retain -- Yes --> Protect[Encrypt, access-control, audit]
  Protect --> Review[Revisit every N months]
  Review --> Retain
```

### Data minimization vs. big-data exploration

| Principle | Big-data exploration | Data minimization |
|---|---|---|
| Collection policy | "We'll find uses later" | "Specified purpose up front" |
| Retention | Indefinite, just in case | Bounded by purpose |
| Joining with other datasets | Encouraged, creative combinations | Only if the new purpose is specified |
| Unforeseen uses | The whole point | Prohibited without fresh basis |
| Risk if breached | Proportional to everything collected | Bounded by what was retained |

See: [[06-data-minimization-and-self-regulation]].

---

## Ethical Checklist for Engineers

Not a compliance ritual — a starting set of questions to force before, during, and after designing a system that handles personal data. Each is drawn from the chapter.

1. **Purpose.** Can I state, in one sentence, the specified and explicit purpose for every field we collect? (GDPR minimum bar.)
2. **Necessity.** Is this data genuinely necessary to provide the service, or only necessary because ads pay for the service?
3. **Consent.** Can a reasonable user understand what is being collected, and can they refuse without losing access to something society now treats as essential?
4. **Feedback loops.** If this model's outputs change the world it predicts on, what does the loop look like? Does it amplify existing inequality or counteract it?
5. **Proxy variables.** Which input features are correlated with protected traits (race, gender, age, disability, belief) even if those traits are not directly used?
6. **Appeal.** If our decision is wrong for an individual, how do they find out and how do they contest it? Is there a human in the loop?
7. **Retention.** What is the purge date for each dataset? Who reviews and enforces it?
8. **Blast radius.** If every record we hold leaked tomorrow — to a criminal, a hostile government, or a future unscrupulous owner — what harm could follow?
9. **Future regimes.** Is this data something we would still want to hold under a government that did not respect human rights?
10. **Accountability.** If this system causes harm, which named human inside our organization is accountable, and do they have authority to stop it?

---

## Trade-offs: Data Utility vs. User Rights

| Dimension | More data / more utility | Less data / stronger rights |
|---|---|---|
| Personalization quality | Higher (more signal) | Lower, but often close to good-enough |
| Model fairness | Opaque; bias can lurk in correlates | Smaller attack surface for bias |
| Breach impact | Catastrophic | Bounded |
| User trust | Erodes once scope is revealed | Compounds over time |
| Regulatory exposure | High (GDPR, sectoral laws) | Low |
| Ability to pivot product | High (exploration is cheap) | Constrained by stated purpose |
| Power concentration | Accumulates in the collector | Remains with users |
| Historical analogy | Pre-regulation industrial factory | Post-regulation factory — slower, sustainable |

The chapter's argument is that the second column is not merely "ethically nicer." It is engineering discipline: the cost of collecting less today is far smaller than the cost of a breach, a scandal, or a regulatory enforcement action tomorrow.

---

## Key Takeaways

1. **Ethics is not a checklist.** It is a participatory, iterative practice of reflection, dialog with affected people, and accountability for outcomes.
2. **Models launder the past into the future.** If you train on discriminatory data, you deploy discrimination at scale with plausible deniability.
3. **Systems thinking beats component thinking.** Reason about the whole loop — users, incentives, deployed decisions — not the isolated model.
4. **Tracking is surveillance when the customer is the advertiser.** Name it honestly; the naming changes the design conversation.
5. **Consent needs to be free, specific, informed, and meaningful.** Clickwrap on an essential service is not consent.
6. **Privacy is a decision right, not a secret.** Surveillance transfers that right from the individual to the collector.
7. **Data is a toxic asset.** Collected data outlives your intentions, your company, and sometimes your government.
8. **Data minimization is concrete engineering.** Collect less, retain less, purge aggressively — and self-regulate before the regulators force it.

---

## See Also

- [[01-algorithmic-decision-making-and-bias]]
- [[02-feedback-loops-and-systems-thinking]]
- [[03-surveillance-as-a-lens]]
- [[04-meaningful-consent-and-privacy]]
- [[05-data-as-toxic-asset]]
- [[06-data-minimization-and-self-regulation]]